# 16 · Monitoring, overload, and measurement

**Level:** advanced · **Time:** 45 minutes · **Prerequisites:** 00, 07, and 10.

Inspect the Management HTTP API, create a visible ready/unacknowledged state, reject excess publications with a queue limit, and measure a small publishing experiment.

The UI and management statistics are sampled and may lag. Correctness assertions below rely on message bodies and protocol responses, not exact dashboard counts. Results are a local teaching experiment, not a production capacity estimate.


In [ ]:
from pathlib import Path
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "course" / "rabbit.py").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from course.rabbit import connect, unique_name, wait_for_message, broker_url
import pika
print("Course root:", ROOT)


## Step 1 — Read the management API

The course's local account can inspect its vhost. Authentication is derived from the broker URL without printing the password. HTTP and AMQP are different endpoints: default local ports are 15672 and 5672 respectively.


In [ ]:
import os
import requests
from urllib.parse import urlsplit, unquote, quote

parts = urlsplit(broker_url())
vhost = unquote(parts.path[1:]) or "/"
management_url = os.getenv("RABBITMQ_MANAGEMENT_URL", "http://localhost:15672").rstrip("/")
http = requests.Session()
http.auth = (unquote(parts.username or ""), unquote(parts.password or ""))

def api_get(path):
    response = http.get(management_url + "/api/" + path, timeout=5)
    response.raise_for_status()
    return response.json()

overview = api_get("overview")
print({key: overview.get(key) for key in ("rabbitmq_version", "cluster_name", "management_version")})


## Step 2 — Hold three deliveries unacknowledged

Open Management → Queues and Streams while this connection remains open. We publish ten messages and receive three without ack. The remaining seven stay ready. The API snapshot may take several seconds to reflect this state; the notebook prints values without asserting their timing.


In [ ]:
queue_name = unique_name("monitor")
connection = connect()
channel = connection.channel()
channel.queue_declare(queue=queue_name)
channel.confirm_delivery()
for number in range(10):
    channel.basic_publish(exchange="", routing_key=queue_name, body=str(number).encode(), mandatory=True)
held = [wait_for_message(channel, queue_name) for _ in range(3)]
info = api_get(f"queues/{quote(vhost, safe='')}/{quote(queue_name, safe='')}")
print("Inspect this queue:", queue_name)
print({key: info.get(key, "not sampled yet") for key in ("messages", "messages_ready", "messages_unacknowledged", "consumers")})
print("This client is holding", len(held), "unacknowledged deliveries")


## Step 3 — Finish the messages and release the connection

If you leave a Pika blocking connection idle between cells for a long time, heartbeat servicing stops; reconnect and redelivery may be required. Run this cell promptly after inspecting the UI.


In [ ]:
try:
    numbers = []
    for method, properties, body in held:
        numbers.append(int(body))
        channel.basic_ack(method.delivery_tag)
    for _ in range(7):
        method, properties, body = wait_for_message(channel, queue_name)
        numbers.append(int(body))
        channel.basic_ack(method.delivery_tag)
    assert sorted(numbers) == list(range(10))
    channel.queue_delete(queue=queue_name)
finally:
    if connection.is_open:
        connection.close()
print("All ten messages completed")


## Step 4 — Reject an excess publication instead of silently growing

This classic queue accepts at most three ready messages and uses `reject-publish` overflow. Publisher confirms let the producer observe the fourth publication's negative acknowledgement. A deployed publisher needs a bounded response to overload; endlessly retrying immediately adds load.


In [ ]:
bounded_queue = unique_name("bounded")
with connect() as connection:
    channel = connection.channel()
    channel.queue_declare(queue=bounded_queue, arguments={"x-max-length": 3, "x-overflow": "reject-publish"})
    channel.confirm_delivery()
    try:
        for number in range(3):
            channel.basic_publish(exchange="", routing_key=bounded_queue, body=str(number).encode(), mandatory=True)
        try:
            channel.basic_publish(exchange="", routing_key=bounded_queue, body=b"excess", mandatory=True)
        except pika.exceptions.NackError:
            print("Expected: broker rejected the excess publication")
        else:
            raise AssertionError("Expected a negative acknowledgement")
    finally:
        channel.queue_delete(queue=bounded_queue)


## Step 5 — Measure exactly what your experiment measures

Compare unconfirmed transient publishing with waiting for a confirm on every message. Both use fresh non-durable queues and 50 tiny messages. The timer covers publishing only; receiving and cleanup are outside it. Unconfirmed time measures client writes, whereas confirmed time also waits for broker acceptance, so these have different guarantees.

For production capacity planning, vary payload size, persistence, queue type, network delay, publish concurrency, consumer work, and failure conditions. Use a dedicated benchmark environment. This is intentionally small enough for a laptop.


In [ ]:
import time

def sample_publish(confirms, count=50):
    name = unique_name("measurement")
    with connect() as connection:
        channel = connection.channel()
        channel.queue_declare(queue=name)
        try:
            if confirms:
                channel.confirm_delivery()
            started = time.perf_counter()
            for number in range(count):
                channel.basic_publish(exchange="", routing_key=name, body=str(number).encode())
            elapsed = time.perf_counter() - started
            received = []
            for _ in range(count):
                method, properties, body = wait_for_message(channel, name)
                received.append(int(body))
                channel.basic_ack(method.delivery_tag)
            assert sorted(received) == list(range(count))
            return {"confirms": confirms, "count": count, "publish_seconds": round(elapsed, 4),
                    "publications_per_second": round(count / max(elapsed, 0.000001))}
        finally:
            channel.queue_delete(queue=name)

measurements = [sample_publish(confirms=value) for value in (False, True)]
for measurement in measurements:
    print(measurement)


## Exercise — Diagnose a slow system

For each observation, name the next measurement you would take:

| Observation | Suggested next check |
|---|---|
| Ready messages grow | Arrival rate versus completion rate; consumer count and downstream latency |
| Unacknowledged messages stay high | In-flight work, stuck handlers, prefetch, and acknowledgement path |
| Connections churn | Heartbeat servicing, connection reuse, retry/backoff, and broker logs |
| Publishers are blocked | Broker memory/disk alarms and flow control |
| Parking queues grow | Failure reason, retry budget, and controlled replay procedure |

Add end-to-end latency by putting a timestamp and event ID in the message and observing when the side effect commits. Do not infer it from publish duration alone.

## Deployment exercise — Security and recovery

The Compose credentials are for a local lab, with ports bound to loopback. For deployment, give each application a distinct user and vhost permissions, use TLS with certificate verification (`amqps://`), inject secrets securely, restrict management access, and practice restore and failover. Never disable certificate verification to make an example connect. Rotation must account for both publishers and consumers. Stream workloads and large replayable logs may call for RabbitMQ Streams; the AMQP queue examples here do not demonstrate that protocol.

**Troubleshooting:** HTTP 401 means authentication failed; 403 means permissions are insufficient; 404 often means a wrong vhost or escaped queue path. RabbitMQ URL path `/learning` names the `learning` vhost; `/` is encoded as `%2F` in API paths.


## Cleanup


In [ ]:
http.close()
print("HTTP session closed; successful experiments have deleted their queues")


## References

- [Management HTTP API](https://www.rabbitmq.com/docs/4.2/http-api-reference)
- [Monitoring](https://www.rabbitmq.com/docs/4.2/monitoring)
- [Queue length limits](https://www.rabbitmq.com/docs/4.2/maxlength)
- [Resource alarms](https://www.rabbitmq.com/docs/4.2/alarms)
- [TLS](https://www.rabbitmq.com/docs/4.2/ssl)
